<a href="https://colab.research.google.com/github/Ibrah-N/PNID_Custom_OCR_Training_Detector_Recognizer/blob/main/PINID_OCR_Classifier_Training.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Data

In [2]:
%cd /content/

/content


In [ ]:
!unzip /content/text_classification.zip

In [11]:
import os
import cv2
import shutil


def rotate_image_90(image):
  return cv2.rotate(image, cv2.ROTATE_90_COUNTERCLOCKWISE)


def copy_image(src_path, dst_path):
  shutil.copy(src_path, dst_path)


def write_image(image, image_path):
  cv2.imwrite(image_path, image)

In [15]:
image_root = "/content/text_classification/images_rot_00"
save_root = "/content/text_classification/images"


os.makedirs(save_root, exist_ok=True)
train_text = "/content/text_classification/train.txt"
val_text = "/content/text_classification/val.txt"

files_list = []
for image_name in os.listdir(image_root):

  src_image_path = os.path.join(image_root, image_name)
  dst_image_path = os.path.join(save_root, image_name)

  copy_image(src_image_path, dst_image_path)

  files_list.append([os.path.join("images", image_name), "0"])

  if not src_image_path.lower().endswith(".jpg"):
    continue

  image = cv2.imread(src_image_path)
  image = rotate_image_90(image=image)

  if image is None:
    print(image, src_image_path)


  dst_image_path = os.path.join(save_root, f"90_{image_name}")
  write_image(image, dst_image_path)

  files_list.append([os.path.join("images", f"90_{image_name}"), "90"])

# Classification Model Training

## Repo & Installation

In [ ]:
!python3 -m pip install paddlepaddle-gpu==3.3.0 -i https://www.paddlepaddle.org.cn/packages/stable/cu130/

Looking in indexes: https://www.paddlepaddle.org.cn/packages/stable/cu130/
     ━━━━━━━━━━━━━━━━━╺━━━━━━━━━━━━━━━━━━━━━━ 0.6/1.4 GB 16.3 MB/s eta 0:00:48

In [2]:
!git clone https://github.com/PaddlePaddle/PaddleX.git

Cloning into 'PaddleX'...
remote: Enumerating objects: 177622, done.
remote: Counting objects: 100% (2824/2824), done.
remote: Compressing objects: 100% (860/860), done.
remote: Total 177622 (delta 2298), reused 2101 (delta 1956), pack-reused 174798 (from 2)
Receiving objects: 100% (177622/177622), 949.75 MiB | 27.52 MiB/s, done.
Resolving deltas: 100% (138038/138038), done.


In [10]:
!pip install ruamel.yaml

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 118.1/118.1 kB 4.9 MB/s eta 0:00:00


In [12]:
!pip install ujson

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 59.1/59.1 kB 3.6 MB/s eta 0:00:00


In [ ]:
!pip install paddleX

In [ ]:
!pip install langchain==1.2.15 langchain-community==0.4.1

In [23]:
!pip install -e .

Obtaining file:///content/PaddleX
  Installing build dependencies ... done
  Checking if build backend supports build_editable ... done
  Getting requirements to build editable ... done
  Preparing editable metadata (pyproject.toml) ... done
  Building editable for paddlex (pyproject.toml) ... done
  Created wheel for paddlex: filename=paddlex-3.5.3.dev0+g898228cf1.d20260512-0.editable-py3-none-any.whl size=21899 sha256=f89c672ddd8b2525becaf3ec0d8920628608ff4a3cea63ca20d58d1bcf8d018e
  Stored in directory: /tmp/pip-ephem-wheel-cache-95pm1h1x/wheels/ef/07/03/5e0d55ec1d7bec8fad5d38007e86fab86c8c32c837dd98c74f
Successfully built paddlex
  Attempting uninstall: paddlex
    Found existing installation: paddlex 3.5.2
    Uninstalling paddlex-3.5.2:
      Successfully uninstalled paddlex-3.5.2


## Data

In [5]:
!wget https://paddle-model-ecology.bj.bcebos.com/paddlex/data/textline_orientation_example_data.tar -P ./dataset
!tar -xf ./dataset/textline_orientation_example_data.tar -C ./dataset/

--2026-05-12 11:54:44--  https://paddle-model-ecology.bj.bcebos.com/paddlex/data/textline_orientation_example_data.tar
Resolving paddle-model-ecology.bj.bcebos.com (paddle-model-ecology.bj.bcebos.com)... 113.137.57.33, 240e:904:2800:11e1:0:ff:b05b:e56c
Connecting to paddle-model-ecology.bj.bcebos.com (paddle-model-ecology.bj.bcebos.com)|113.137.57.33|:443... connected.
HTTP request sent, awaiting response... 200 OK
Length: 9492480 (9.1M) [application/octet-stream]
Saving to: ‘./dataset/textline_orientation_example_data.tar’

textline_orientatio 100%[===================>]   9.05M  1.07MB/s    in 35s     

2026-05-12 11:55:22 (263 KB/s) - ‘./dataset/textline_orientation_example_data.tar’ saved [9492480/9492480]



In [17]:
!python main.py -c paddlex/configs/modules/textline_orientation/PP-LCNet_x0_25_textline_ori.yaml \
    -o Global.mode=check_dataset \
    -o Global.dataset_dir=./dataset/textline_orientation_example_data

Connecting to https://paddle-model-ecology.bj.bcebos.com/paddlex/PaddleX3.0/fonts/PingFang-SC-Regular.ttf ...
[==================================================] 100.00%
Check dataset passed !


## Model Training

In [2]:
!wget https://paddle-model-ecology.bj.bcebos.com/paddlex/official_pretrained_model/PP-LCNet_x0_25_textline_ori_pretrained.pdparams

--2026-05-12 12:11:49--  https://paddle-model-ecology.bj.bcebos.com/paddlex/official_pretrained_model/PP-LCNet_x0_25_textline_ori_pretrained.pdparams
Resolving paddle-model-ecology.bj.bcebos.com (paddle-model-ecology.bj.bcebos.com)... 103.235.47.176, 240e:904:2800:11e1:0:ff:b05b:e56c
Connecting to paddle-model-ecology.bj.bcebos.com (paddle-model-ecology.bj.bcebos.com)|103.235.47.176|:443... connected.
HTTP request sent, awaiting response... 200 OK
Length: 994943 (972K) [application/octet-stream]
Saving to: ‘PP-LCNet_x0_25_textline_ori_pretrained.pdparams’

PP-LCNet_x0_25_text 100%[===================>] 971.62K  71.8KB/s    in 16s     

2026-05-12 12:12:07 (60.5 KB/s) - ‘PP-LCNet_x0_25_textline_ori_pretrained.pdparams’ saved [994943/994943]



In [1]:
%cd /content/PaddleX
!python main.py -c paddlex/configs/modules/textline_orientation/PP-LCNet_x0_25_textline_ori.yaml \
    -o Global.mode=train \
    -o Global.dataset_dir=./dataset/textline_orientation_example_data

/content/PaddleX
/usr/local/lib/python3.12/dist-packages/paddle/utils/cpp_extension/extension_utils.py:712: UserWarning: No ccache found. Please be aware that recompiling all source files may be required. You can download and install ccache from: https://github.com/ccache/ccache/blob/master/doc/INSTALL.md
  warnings.warn(warning_message)
A new field (uniform_output_enabled) detected!
A new field (pdx_model_name) detected!
A new field (export_with_pir) detected!
['/usr/bin/python3', 'tools/train.py', '-c', '/root/.paddlex/tmpvltvp8z5/clsmodel_PP-LCNet_x0_25_textline_ori.yml', '-o', 'Global.eval_during_train=True']

Log path: /content/PaddleX/output/train.log 

/usr/local/lib/python3.12/dist-packages/paddle/utils/cpp_extension/extension_utils.py:712: UserWarning: No ccache found. Please be aware that recompiling all source files may be required. You can download and install ccache from: https://github.com/ccache/ccache/blob/master/doc/INSTALL.md
  warnings.warn(warning_message)
[2026/05/

In [14]:
!paddlex --install PaddleClas

Now download and update the repos: ['PaddleClas'].
Connecting to https://paddle-model-ecology.bj.bcebos.com/paddlex/PaddleX3.0/repos/PaddleClas.tar ...
[==================================================] 100.00%
Extracting PaddleClas.tar
[==================================================] 100.00%
remote: Total 0 (delta 0), reused 0 (delta 0), pack-reused 0 (from 0)
From https://github.com/PaddlePaddle/PaddleClas
 * branch            develop    -> FETCH_HEAD
HEAD is now at e75732b update pre-commit (#3406)
All repos are existing.
Dependencies are listed below:
# PaddleClas dependencies
prettytable
ujson
opencv-contrib-python
pillow>=9.0.0
tqdm
PyYAML>=5.1
visualdl>=2.2.0
scipy>=1.0.0
scikit-learn>=0.21.0
gast==0.3.3
faiss-cpu
easydict
numpy==1.24.4; python_version<"3.12"
numpy==1.26.4; python_version>="3.12"
packaging
Now installing the packages...
Ignoring numpy: markers 'python_version < "3.12"' don't match your environment
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 61.0/61.0 kB 

## Model Testing